# STEER presentation notebook — story, evidence, and edits

**Purpose:** prepare a short presentation that starts with our Cary/Morrisville/Apex experience, explores composite-score variation, shows why composites can conceal different profiles, and explains the move to five-score comparison. This notebook is a scaffold: verify local selections and source claims before presenting.

**Story spine**
1. Who we are and the nearby communities our friends and siblings experienced.
2. Our question: how different do nearby school attendance boundary (SAB) communities seem?
3. What composite scores show—and what they leave out.
4. Yuma example: similar composites in one district, contrasting domain profiles.
5. Plain-language explanation of comparing five standardized scores together.
6. What this helps people ask, and what it cannot establish.

ODIS measures describe community stress indicators associated with SABs; they are not school performance measures. Keep interpretation descriptive.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px

# Notebook-safe import when launched from the repository root or notebooks/
ROOT = Path.cwd()
if not (ROOT / 'data.py').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from data import load_dataset, DOMAINS, COMPOSITE

schools = load_dataset()
print(f'{len(schools):,} records; domains: {", ".join(DOMAINS)}')

## 1. Composite score: start with a visible pattern

**Presenter wording (edit):** “We grew up around Cary, Morrisville, and Apex, and our friends and older siblings experienced different nearby communities and high schools. We wondered how different those places might be. We started by comparing the composite score.”

The next cells give four possible visuals. Choose the ones that best support the spoken story. City names are only a convenient grouping; city and SAB boundaries do not necessarily align. Add your agreed local SAB/school selection once the team confirms it.

In [ ]:
# Local exploratory subset; edit only after agreeing which records represent your story.
LOCAL_CITIES = ['Cary', 'Morrisville', 'Apex', 'Raleigh', 'Durham', 'Chapel Hill']
local = schools.loc[schools.State.eq('NC') & schools.City.isin(LOCAL_CITIES)].copy()
print(f'{len(local)} exact city-label matches across {local.City.nunique()} cities')
local[['Name', 'City', COMPOSITE, *DOMAINS]].head()

In [ ]:
# Visual 1: distribution and range of composite scores in selected Triangle cities
fig = px.box(local, x='City', y=COMPOSITE, points='all', color='City',
             category_orders={'City': LOCAL_CITIES}, title='Composite scores across selected Triangle cities')
fig.update_layout(showlegend=False, xaxis_title='City', yaxis_title='Composite score')
fig.show()

In [ ]:
# Visual 2: five-domain city averages (descriptive; interpret with the boundary caveat)
city_domains = (local.groupby('City', observed=True)[list(DOMAINS)].mean()
                .reindex(LOCAL_CITIES).dropna(how='all').reset_index()
                .melt('City', var_name='Domain', value_name='Mean score'))
px.line(city_domains, x='City', y='Mean score', color='Domain', markers=True,
        title='Average five-domain profile across selected Triangle cities').show()

In [ ]:
# Visual 3: standardize each domain across this local comparison group, then
# calculate each record's within-profile SD. The result depends on this group.
from sklearn.preprocessing import StandardScaler
z = pd.DataFrame(StandardScaler().fit_transform(local[list(DOMAINS)]),
                  columns=DOMAINS, index=local.index)
local['Profile spread (σ)'] = z.std(axis=1, ddof=0)
px.scatter(local, x=COMPOSITE, y='Profile spread (σ)', color='City', hover_name='Name',
           title='Composite score versus standardized five-domain spread').show()

## 2. Similar composites, very different profiles: Yuma Union High School District

Gila Ridge High School (Yuma) and San Luis High School (San Luis) are both in Yuma Union High School District. Their composite scores are only 6 points apart. The district describes San Luis as about 20 miles from Yuma ([district history](https://www.yumaunion.org/welcome/welcome/our-history)).

| School | City | Composite | Economic | Education | Health | Housing | Crime |
|---|---|---:|---:|---:|---:|---:|---:|
| Gila Ridge High School | Yuma | 36 | 41 | 29 | 26 | 55 | 29 |
| San Luis High School | San Luis | 42 | 53 | 90 | 33 | 6 | 29 |

Education differs by 61 points and Housing by 49. This is a concrete case where similar composites and a shared district label conceal sharply different domain profiles. These ODIS scores describe community conditions associated with attendance boundaries, not school performance. The district-distance statement comes from its [history page](https://www.yumaunion.org/welcome/welcome/our-history).


In [ ]:
pair_names = ['Gila Ridge High School', 'San Luis High School']
pair = schools.loc[schools.Name.isin(pair_names)].copy()
assert len(pair) == 2, 'Confirm the school names in the current data release.'
assert pair['School District'].nunique() == 1
assert pair[COMPOSITE].max() - pair[COMPOSITE].min() == 6
pair[['Name', 'City', 'School District', COMPOSITE, *DOMAINS]]

In [ ]:
# Visual 4: compare the five underlying values directly
pair_long = pair.melt(id_vars=['Name', 'City', COMPOSITE], value_vars=list(DOMAINS),
                      var_name='Domain', value_name='Score')
pair_long['School'] = pair_long['Name'] + ' · ' + pair_long['City']
px.bar(pair_long, x='Domain', y='Score', color='School', barmode='group',
       title='Yuma Union: composite scores 6 points apart, contrasting domain profiles').show()

## 3. How we compared profiles—in plain language

For each selected domain, subtract the comparison group’s average and divide by that domain’s typical spread. This puts the five number scales on a comparable footing for this calculation. Then measure the straight-line distance between the two five-number points: square each gap, add them, and take the square root.

A larger gap in one domain contributes more to the total distance. The nearest profile records have the smallest distances under the selected domains and comparison group. Changing the comparison group can change the standardized values and neighbors. This is a retrieval aid, not proof of shared causes, needs, or effective policies.

**One-line visual / whiteboard:** `(score − group average) ÷ group spread` for each domain → `√(gap₁² + gap₂² + … + gap₅²)`

### Policy question
If decision-makers treat two places as similar because their composite scores match, what domain-level differences might be overlooked? The tool helps make that question inspectable; it does not prescribe a funding or collaboration decision.

## 4. Final checks before presenting

- Replace or confirm the Triangle city selection with the SAB/high-school examples the team intends to discuss.
- Confirm the source release, shared Yuma Union district label, and Yuma pair scores against the loaded dataset.
- Avoid saying composite scores alone prove local areas are “polarized”; describe the observed score spread precisely.
- Explain that ODIS scores characterize community conditions, not school performance.
- Keep standardization, candidate group, missing-score handling, and descriptive limits visible when explaining STEER.
- Do not infer causation, intervention effects, grant eligibility, or that nearest profiles are validated twins.

**Close (draft):** “We built STEER to make the pieces behind a composite easier to inspect. A useful next step is to ask data owners and community stakeholders whether these comparisons are meaningful for a real decision.”